# Pearl-Chat: Native Luganda Language Model from Scratch in Pure JAX

Workshop walkthrough notebook for PyCon Africa 2026.
This notebook covers environment verification, BPE tokenizer inspection, explicit state representation in Flax NNX, training, sharding, and inference.

## Phase 0: Environment setup and device inspection

In [ ]:
import jax
import flax.nnx as nnx
import optax
import orbax.checkpoint as ocp
import grain.python as grain

print("JAX backend:", jax.default_backend())
print("Available devices:", jax.devices())

## Phase 1: Luganda BPE tokenizer

In [ ]:
from pearlchat.tokenizer import LugandaTokenizer

tokenizer = LugandaTokenizer.load("data/tokenizer")
print(f"Vocabulary size: {tokenizer.vocab_size}")

sample = "Abayizi bagenda ku masomero okufuna amagezi n'obukugu."
tokens = tokenizer.encode(sample)
print("Original text:", sample)
print("Encoded tokens:", tokens)
print("Decoded text:", tokenizer.decode(tokens))

## Phase 2: Data pipeline with Grain

In [ ]:
from pearlchat.data_pipeline import LugandaTextDataSource, create_data_loader

source = LugandaTextDataSource.from_file("data/processed/train.txt", tokenizer=tokenizer, context_length=128)
loader = create_data_loader(source, batch_size=4, seed=42, shuffle=True)
batch = next(iter(loader))
print("Batch inputs shape:", batch["inputs"].shape)
print("Batch targets shape:", batch["targets"].shape)

## Phase 3: Explicit state in Flax NNX versus PyTorch implicit state

Side by side conceptual comparison:

### Implicit state (PyTorch style pseudocode):


### Explicit state (Flax NNX graph decomposition):


In [ ]:
from pearlchat.config import ModelConfig
from pearlchat.model import PearlChatModel

config = ModelConfig(vocab_size=tokenizer.vocab_size, context_length=128, embed_dim=128, num_heads=4, num_layers=4, feed_forward_dim=512)
rngs = nnx.Rngs(params=42, dropout=42)
model = PearlChatModel(config, rngs)

print(f"Total parameters: {model.count_parameters():,}")
graphdef, state = nnx.split(model)
print("State leaves count:", len(jax.tree_util.tree_leaves(state)))

## Phase 4: Training step and live loss decrease

In [ ]:
from pearlchat.checkpointing import CheckpointManager
from pearlchat.config import TrainConfig
from pearlchat.train import run_training_loop

train_config = TrainConfig(batch_size=8, total_steps=20, eval_every=5, learning_rate=5e-4)
ckpt_manager = CheckpointManager("checkpoints/pearlchat-warm-start")

# Run short demonstration training
final_loss = run_training_loop(
    model=model,
    train_config=train_config,
    train_loader=loader,
    checkpoint_manager=ckpt_manager,
    start_step=0,
    target_steps=20,
)

## Phase 5: Sharding demonstration across device mesh

In [ ]:
from pearlchat.sharding import create_device_mesh, get_data_parallel_sharding, get_model_parallel_sharding, print_sharding_diagram, shard_array
import jax.numpy as jnp

mesh = create_device_mesh(mesh_shape=(4, 2))
print("Device mesh:", mesh)

data_sharding = get_data_parallel_sharding(mesh)
inputs = jnp.ones((16, 64), dtype=jnp.float32)
sharded_inputs = shard_array(inputs, data_sharding)
print_sharding_diagram(sharded_inputs, title="Data-parallel input tensor")

## Phase 6: Autoregressive text generation

In [ ]:
from pearlchat.generate import generate_text

prompt = "Oli otya?"
response = generate_text(model, tokenizer, prompt=prompt, max_new_tokens=32, temperature=0.7)
print(f"Prompt: {prompt}")
print(f"Response: {response}")